### Selection strategy

Three buckets of examples to mix, totalling 10-12:
- Bucket A — Clean ladder cases (3-4 examples)
Thoughts where E1 < E2 < E3 < E4 in BERTScore F1 (or at least E1 < E4 with steady improvement). These illustrate the "best case" of the intervention working as hypothesised and let the chapter show the reader a positive concrete example.
- Bucket B — Metric/quality divergence cases (3-4 examples)
Thoughts where E4 has the lowest BERTScore but appears qualitatively strong, OR where E1 has the highest BERTScore but appears qualitatively weak. These support the methodological argument that automatic metrics are limited.
- Bucket C — Ladder failures (2-3 examples)
Thoughts where E4 scores worst, or where category-matching seems to hurt. These illustrate failure modes honestly and let the chapter discuss when the prompting strategies do not help.

In [1]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("../../models/06_reframing/results")
df = pd.read_csv(RESULTS_DIR / "experiment_outputs_scored_v2.csv")

# Pivot to wide: one row per thought_id, columns for each experiment's BERTScore
wide = df.pivot_table(
    index=["thought_id", "category", "situation", "thought", "reference_reframe"],
    columns="experiment_id",
    values="bertscore_f1",
).reset_index()

# Order columns
wide = wide[["thought_id", "category", "situation", "thought", "reference_reframe", "E1", "E2", "E3", "E4"]]

# Add derived features for selection
wide["e4_minus_e1"] = wide["E4"] - wide["E1"]
wide["e4_minus_e3"] = wide["E4"] - wide["E3"]
wide["is_monotonic"] = (
    (wide["E2"] >= wide["E1"]) &
    (wide["E3"] >= wide["E2"]) &
    (wide["E4"] >= wide["E3"])
)
wide["max_exp"] = wide[["E1", "E2", "E3", "E4"]].idxmax(axis=1)
wide["min_exp"] = wide[["E1", "E2", "E3", "E4"]].idxmin(axis=1)

print(f"Total test thoughts: {len(wide)}")
print(f"Strictly monotonic ladder (E1<E2<E3<E4): {wide['is_monotonic'].sum()}")
print(f"E4 is the best: {(wide['max_exp']=='E4').sum()}")
print(f"E1 is the best: {(wide['max_exp']=='E1').sum()}")
print(f"E4 is the worst: {(wide['min_exp']=='E4').sum()}")

Total test thoughts: 57
Strictly monotonic ladder (E1<E2<E3<E4): 7
E4 is the best: 26
E1 is the best: 13
E4 is the worst: 11


In [4]:
# Bucket A — clean ladder cases (largest E4-E1 gap, monotonic preferred)
bucket_a = wide.sort_values(["is_monotonic", "e4_minus_e1"], ascending=[False, False]).head(8)
print("=== Bucket A candidates (clean ladder, large E4-E1 gap) ===")
print(bucket_a[["thought_id", "category", "thought", "E1", "E2", "E3", "E4", "e4_minus_e1", "is_monotonic"]].to_string(index=False))

=== Bucket A candidates (clean ladder, large E4-E1 gap) ===
 thought_id                   category                                                                                                                       thought       E1       E2       E3       E4  e4_minus_e1  is_monotonic
          3                   labeling                                                                                                       he took me for granted! 0.150126 0.179575 0.188277 0.491756     0.341631          True
         14                   labeling                                                                  I'm not who I want to be. I hate my appearance and my voice. 0.271285 0.384217 0.386051 0.412107     0.140822          True
        177            catastrophizing                                                                                           I'm afraid she's going to die soon. 0.170248 0.198983 0.244573 0.304666     0.134418          True
        213               mi

In [5]:
# Bucket B — metric/quality divergence candidates
# These are flagged for manual inspection; we don't know which are "qualitatively strong despite low score" until we read them
# Strategy: pick thoughts where E4 has the WORST score (despite being our headline approach)
# AND thoughts where E1 has the BEST score by a wide margin
bucket_b_e4_worst = wide[wide["min_exp"] == "E4"].sort_values("E4").head(5)
bucket_b_e1_best = wide[wide["max_exp"] == "E1"].sort_values("E1", ascending=False).head(5)

print("\n=== Bucket B candidates: E4 is worst ===")
print(bucket_b_e4_worst[["thought_id", "category", "thought", "E1", "E2", "E3", "E4"]].to_string(index=False))
print("\n=== Bucket B candidates: E1 is best ===")
print(bucket_b_e1_best[["thought_id", "category", "thought", "E1", "E2", "E3", "E4"]].to_string(index=False))


=== Bucket B candidates: E4 is worst ===
 thought_id                category                               thought       E1       E2       E3       E4
        105         catastrophizing                That I'm going to fail 0.185403 0.234204 0.223624 0.121186
         71 all-or-nothing thinking  why can't I ever get anything I want 0.359555 0.345569 0.364895 0.173235
        126       should statements      I shouldn't have changed my hair 0.270890 0.256533 0.259006 0.188076
        170 all-or-nothing thinking I can't bring myself to be motivated. 0.217004 0.230019 0.222312 0.192464
        138 all-or-nothing thinking          Medication can be dangerous. 0.256685 0.254734 0.199841 0.193509

=== Bucket B candidates: E1 is best ===
 thought_id            category                                                                       thought       E1       E2       E3       E4
        107     fortune telling       People will never perceive me as acting naturally in social situations. 0

In [6]:
# Bucket C — ladder failures (E4 lower than E1 or E3)
bucket_c = wide[wide["e4_minus_e1"] < 0].sort_values("e4_minus_e1").head(5)
print("\n=== Bucket C candidates (E4 worse than E1) ===")
print(bucket_c[["thought_id", "category", "thought", "E1", "E2", "E3", "E4", "e4_minus_e1"]].to_string(index=False))


=== Bucket C candidates (E4 worse than E1) ===
 thought_id                   category                                  thought       E1       E2       E3       E4  e4_minus_e1
         71    all-or-nothing thinking     why can't I ever get anything I want 0.359555 0.345569 0.364895 0.173235    -0.186319
         47            fortune telling               She will never love me now 0.355980 0.281989 0.269458 0.219419    -0.136561
        126          should statements         I shouldn't have changed my hair 0.270890 0.256533 0.259006 0.188076    -0.082815
         45 disqualifying the positive My grades are important and I did poorly 0.312633 0.312633 0.229763 0.232810    -0.079823
         77                   labeling               He's so damn inconsiderate 0.304135 0.253134 0.254603 0.238250    -0.065886


In [7]:
# Final selection: assemble the 10-12 with category coverage constraint
# Tentative: 4 from A, 4 from B (mix of e4_worst and e1_best), 3 from C
# But enforce >=5 distinct categories

def pick_with_coverage(candidates_list, n_total, min_categories=5):
    """
    Greedy selection: pick from candidate lists in priority order,
    preferring categories not yet seen until min_categories is met.
    """
    picked = []
    picked_ids = set()
    picked_cats = set()

    # First pass: prefer new categories
    for candidates in candidates_list:
        for _, row in candidates.iterrows():
            if len(picked) >= n_total:
                break
            if row["thought_id"] in picked_ids:
                continue
            if row["category"] not in picked_cats or len(picked_cats) >= min_categories:
                picked.append(row)
                picked_ids.add(row["thought_id"])
                picked_cats.add(row["category"])
        if len(picked) >= n_total:
            break

    # Second pass: fill remaining slots if needed (ignore coverage)
    if len(picked) < n_total:
        for candidates in candidates_list:
            for _, row in candidates.iterrows():
                if len(picked) >= n_total:
                    break
                if row["thought_id"] not in picked_ids:
                    picked.append(row)
                    picked_ids.add(row["thought_id"])

    return pd.DataFrame(picked).reset_index(drop=True), picked_cats

# Targets: 4 from A, 4 from B (split), 3 from C
final, covered_cats = pick_with_coverage(
    [bucket_a.head(4), bucket_b_e4_worst.head(2), bucket_b_e1_best.head(2), bucket_c.head(3)],
    n_total=11,
    min_categories=5,
)

print(f"\n=== FINAL SELECTION ({len(final)} examples, {len(covered_cats)} categories) ===")
print(f"Categories covered: {sorted(covered_cats)}")
print()
print(final[["thought_id", "category", "thought", "E1", "E2", "E3", "E4", "e4_minus_e1"]].to_string(index=False))


=== FINAL SELECTION (10 examples, 7 categories) ===
Categories covered: ['all-or-nothing thinking', 'catastrophizing', 'fortune telling', 'labeling', 'mind reading', 'personalizing', 'should statements']

 thought_id                category                                                                 thought       E1       E2       E3       E4  e4_minus_e1
          3                labeling                                                 he took me for granted! 0.150126 0.179575 0.188277 0.491756     0.341631
        177         catastrophizing                                     I'm afraid she's going to die soon. 0.170248 0.198983 0.244573 0.304666     0.134418
        213            mind reading                                              I'm not attractive enough. 0.126658 0.197489 0.199130 0.258994     0.132337
         71 all-or-nothing thinking                                    why can't I ever get anything I want 0.359555 0.345569 0.364895 0.173235    -0.186319
        1

In [8]:
# Save the selection plus full prompt/output data for each picked thought
# This is what will become Table X in Section 5.6 of the thesis
selected_ids = set(final["thought_id"])
qualitative = df[df["thought_id"].isin(selected_ids)].copy()

# Order by experiment then thought_id for the table view
qualitative = qualitative.sort_values(["thought_id", "experiment_id"])

qual_path = RESULTS_DIR / "qualitative_selection.csv"
qualitative.to_csv(qual_path, index=False)
print(f"Saved: {qual_path}")
print(f"Rows: {len(qualitative)} (= {len(selected_ids)} thoughts × 4 experiments)")

Saved: ../../models/06_reframing/results/qualitative_selection.csv
Rows: 40 (= 10 thoughts × 4 experiments)
